# Project 1 — Point-in-Time ECL Engine
End-to-end walkthrough of the Vasicek-calibrated, dynamically-staged
Ind AS 109 expected credit loss engine. All numbers in this notebook
are produced by `src/pipeline.py`; the entire pipeline runs with
`python scripts/run_all.py`.


## 1. Synthetic retail loan book
**Schema.** Each row is a single loan. `int_rate`, `loan_amnt`, `grade`,
`annual_inc`, `dti`, `delinq_2yrs`, `inq_last_6mths`, `revol_util`,
`home_ownership`, `purpose` mirror public LendingClub disclosure.
`default_12m` is the 12-month default flag and `origination_z` is the
Vasicek systematic factor at origination (leakage-free; we never feed
this column into the classifier).


In [ ]:
import pandas as pd
df = pd.read_csv('data/synthetic_loans.csv')
print('Shape:', df.shape)
print('Default rate (%):', round(100 * df['default_12m'].mean(), 2))
print('Grade distribution:')
print(df['grade'].value_counts().sort_index())
df.head(3)


## 2. Macroeconomic series and default rates
`synthetic_macro.csv` contains monthly observations of GDP growth,
CPI, repo rate, unemployment, plus the latent systematic factor
`z_systematic` and the implied `default_rate`.


In [ ]:
import pandas as pd
macro = pd.read_csv('data/synthetic_macro.csv', parse_dates=['month'])
macro[['month','default_rate','gdp_growth','repo_rate','unemployment']].head()


![Macro drivers](../reports/figures/fig_macro_drivers.png)

![Z systematic](../reports/figures/fig_z_systematic.png)


## 3. Vasicek model and Ornstein-Uhlenbeck dynamics
Closed-form inversion yields the systematic factor Z_t. Asset correlation
ρ is calibrated by moment-matching default-rate volatility to the
Vasicek identity. The mean-reverting Z_t dynamics are estimated by
exact MLE on the discretised AR(1) form.

- Asset correlation ρ = **0.0500**
- Long-run PD_PiT = **0.0500**
- OU θ = **3.1546**   μ = **-0.1720**   σ = **1.6202**


![Z distribution](../reports/figures/fig_z_distribution.png)


## 4. Forward OU simulation — baseline vs adverse
200 Monte-Carlo paths simulated for 60 monthly steps (5y) under
the calibrated OU process. The adverse scenario overrides both
the starting state and the long-run mean to model a lasting
regime shift.


![OU paths](../reports/figures/fig_ou_paths.png)


### PD term structure (path-mean across simulations)
- Baseline: **[0.0511, 0.0521, 0.051, 0.0518, 0.0532]**
- Adverse:  **[0.0979, 0.1001, 0.098, 0.0997, 0.1035]**

The Vasicek conditional PD translates each year's mean Z into a
yearly marginal PD, giving the 5-year point-in-time term structure.


![Term structure](../reports/figures/fig_pd_term_structure.png)


### Sample cohort diagnostic (from the Ind AS 109 brief)
Reproducing the marginal-to-cumulative PD identity from the brief:
$\Pi_{t}(1 - {}_{t-1}q_t) = (1 - {}_0q_T)$.


In [ ]:
from src.pd_term_structure import term_structure_diagnostic
diag = term_structure_diagnostic()
for y, m, c in zip([1,2,3], diag['marginal'], diag['cumulative']):
    print(f'  Year {y}: marginal={m:.4f}  cumulative={c:.4f}')


## 5. LightGBM classifier with Platt scaling
Classifier is trained on a leakage-clean feature set with time-series
split. Platt scaling maps raw scores to calibrated probabilities.

- Test AUC (raw):  **0.6059**
- Test AUC (Platt):**0.6059**
- Brier (raw):     **0.0920**
- Brier (Platt):   **0.0923**


![Classifier outputs](../reports/figures/fig_classifier_outputs.png)


## 6. SICR-driven staging
Each loan is assigned to Stage 1, 2 or 3 using the rule:

- Stage 3 if DPD > 90 days (defaulted at reporting date).
- Stage 2 if 30 < DPD ≤ 90, **or** PiT-LifetimePD / OriginationPD > 1.5
  (the SICR criterion that replaces static DPD triggers).
- Stage 1 otherwise.


In [ ]:
from src.staging import stage_breakdown, STAGE1, STAGE2, STAGE3
out_df = pd.read_csv('data/ecl_output.csv')
print('Stage mix (baseline):', stage_breakdown(out_df['stage_baseline'].values))
print('Stage mix (adverse) :', stage_breakdown(out_df['stage_adverse'].values))


## 7. ECL aggregation per stage and scenario
For Stage 1 we compute the 12-month ECL (sum of T=1 year PD · LGD · EAD · D).
For Stage 2/3 we run the lifetime (T=5 year) term structure with the
stressed-LGD uplift applied to Stage 3.


![ECL per stage](../reports/figures/fig_ecl_per_stage.png)


## 8. Stress test — forecast-error reduction
We measure how well each candidate engine anticipates the realised
first-year portfolio losses under the adverse scenario. Coverage
ratio = mean(provisions) / mean(realised losses).

- PiT mean provisions (adverse) : INR **6,248,282**
- Realised adverse losses (mean) : INR **4,934,505**
- Static-TTC provisions          : INR **2,262,861**
- PiT coverage ratio (adverse)   : **1.266** (>1 = over-provisioned, regulatorily safe)
- TTC coverage ratio (adverse)   : **0.459** (<1 = under-provisioned, regulatorily risky)
- **Forecast-error reduction**   : **50.83%**


![Stress coverage](../reports/figures/fig_stress_coverage.png)


## 9. Audit trail
Every run produces `reports/audit_trail.json` with hashed model
manifests, calibrated parameters, ECL totals, and the Ind AS 109
staging counts. This is what you would table during a regulatory
review meeting.


In [ ]:
import json
with open('reports/audit_trail.json') as f:
    audit = json.load(f)
print(json.dumps({k: v for k, v in audit.items() if k not in ('data_hashes',)}, indent=2, default=str)[:1800])
